# Heavy Computation (WSI example)

This page demonstrates how documentation can include **heavy operations**
that run on a remote server. In a real project you might load a whole-slide
image (WSI) here — the reader's browser never has to handle the data.

Click 🚀 → **Live Code** to activate remote execution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import sleep, time

## Simulated WSI loading

The cell below **simulates** loading a large whole-slide image. In your
real documentation you would replace this with actual library calls, e.g.:

```python
from rationai.data import load_wsi
wsi = load_wsi("/data/slides/sample.svs")
```

Because the kernel runs on a remote server (Binder / JupyterHub), the
multi-GB file is loaded server-side. Only the final visualization is sent
to the browser.

In [ ]:
# --- Simulated heavy WSI load ---
print("Loading whole-slide image (simulated)...")
start = time()
sleep(2)  # simulate I/O latency

# Create a fake tissue thumbnail (256x256 RGB)
np.random.seed(42)
thumbnail = np.random.randint(180, 240, (256, 256, 3), dtype=np.uint8)
# Add some "tissue" blobs
for _ in range(30):
    cx, cy = np.random.randint(20, 236, 2)
    r = np.random.randint(10, 40)
    yy, xx = np.ogrid[-cx:256-cx, -cy:256-cy]
    mask = xx**2 + yy**2 <= r**2
    thumbnail[mask] = np.random.randint(120, 180, 3, dtype=np.uint8)

elapsed = time() - start
print(f"WSI loaded in {elapsed:.1f}s  (shape: 100000x80000 px, shown as thumbnail)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(thumbnail)
ax.set_title("WSI Thumbnail (server-side rendering)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Running a tile-level inference

Below we simulate extracting tiles and running inference — another operation
that would be prohibitively expensive in-browser but works fine on a remote
kernel.

In [ ]:
# Simulate tile extraction and inference
n_tiles = 64
tile_size = 256
print(f"Extracting {n_tiles} tiles of {tile_size}x{tile_size} px ...")

scores = []
for i in range(n_tiles):
    tile = np.random.randn(tile_size, tile_size, 3)  # fake tile
    score = 1 / (1 + np.exp(-tile.mean()))            # fake sigmoid score
    scores.append(score)

scores = np.array(scores)
print(f"Inference complete.  Mean score: {scores.mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 3))
heatmap = scores.reshape(8, 8)
im = ax.imshow(heatmap, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_title("Tile-level prediction heatmap")
plt.colorbar(im, ax=ax, label="Score")
plt.tight_layout()
plt.show()